# Phase 4b: tune the model for dark skin, one change at a time, with a locked test set

**Decision-support prototype only. Not a diagnosis. Consult a qualified clinician.**

Phase 4 (run 1) fine-tuned the model on dark-skin clinical photos from Fitzpatrick17k (Fitzpatrick IV-VI) and measured it by cross-validation. This notebook tries to improve on that recipe, honestly:

1. **Locked test set.** 20% of the dark-skin images (stratified by class) and half of the ISIC benign dark-skin patients are put aside first. They are used **once**, at the very end, and never to choose anything.
2. **Development set.** The other 80% of dark-skin images, and the other half of the ISIC benign patients, are used for tuning by 4-fold cross-validation: every development image is predicted by a model that did not train on it (benign lesions are split by patient).
Run 1 raised dark-skin accuracy a lot but also raised false alarms on benign dark-skin lesions from 1.7% to 38%. The usable Fitzpatrick17k dark-skin images are almost all cancers (BCC and melanoma), so the model learned a shortcut: "dark skin means cancer". The main fix tried here is to also train on **benign dark-skin lesions** from the ISIC Archive (development patients only). These have no lesion type, only "benign", so they use a simple loss that pushes probability away from the three cancer / pre-cancer classes.

3. **One change at a time.** Starting from the run-1 recipe, the notebook tries a fixed list of changes in a fixed order. A change is kept only if it raises the development dark-skin macro-F1 by at least 0.005 **and** does not hurt HAM10000 validation (macro-F1 drop of at most 0.01) **and** does not raise false alarms on held-out development ISIC benign patients by more than 0.02. A recipe that meets these safety limits always beats one that does not. Every trial is logged.
4. **Final model.** The best recipe is trained on HAM10000 plus all development images and tested once on the locked sets and the original HAM10000 test set, against the original model.

**Honest scope.** The dark-skin data is small and web-sourced (no patient ids, some label noise). The locked test set is small, so its confidence intervals are wide. Nothing here shows the model is safe to use on dark skin.

Runs on **Kaggle**: GPU T4, internet on, dataset `kmader/skin-cancer-mnist-ham10000` attached.

## 1. Setup

In [ ]:
import os, sys, io, json, time, random, subprocess, urllib.request, urllib.parse
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image, ImageFile
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score
ImageFile.LOAD_TRUNCATED_IMAGES = True

IN_KAGGLE = os.path.exists("/kaggle/working")
SMOKE = os.environ.get("PHASE4_SMOKE") == "1"      # tiny local test mode

# ---------------- ONE CONFIG BLOCK: every number lives here ----------------
CFG = dict(
    repo_url="https://github.com/Elvis-Kayonga/ai-skin-lesion-detection-system-capstone-project.git",
    seed=42,
    class_codes=["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"],
    high_risk=["akiec", "bcc", "mel"],          # pre-cancer / cancer classes
    input_size=300, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
    dark_fitz_types=[4, 5, 6],   # Fitzpatrick IV-VI
    lock_share=0.2,              # share of dark-skin images locked away as the final test set
    dev_folds=4,                 # cross-validation folds inside the development set
    batch_size=32, samples_per_epoch=3000, weight_decay=1e-4,
    # run-1 recipe = starting point of the tuning loop
    base=dict(epochs=6, dark_share=0.3, fitz3_share=0.0, benign_share=0.0, lr_backbone=1e-4, lr_head=3e-4,
              label_smoothing=0.0, freeze_blocks=0, dark_class_balance=False),
    # changes tried in this order, each on top of the best recipe so far
    trials=[("add benign dark-skin lesions (0.15)", dict(benign_share=0.15)),
            ("more benign dark-skin share (0.3)",  dict(benign_share=0.3)),
            ("more dark-skin share (0.45)",        dict(dark_share=0.45)),
            ("longer training (10 epochs)",       dict(epochs=10)),
            ("class-balanced dark sampling",      dict(dark_class_balance=True)),
            ("label smoothing 0.1",               dict(label_smoothing=0.1)),
            ("add Fitzpatrick III photos (0.15)", dict(fitz3_share=0.15)),
            ("freeze first 4 backbone blocks",    dict(freeze_blocks=4)),
            ("lower backbone learning rate",      dict(lr_backbone=3e-5))],
    min_gain=0.005, time_budget_h=8.0,
    ham_val_f1_drop=0.01, dev_false_alarm_rise=0.02,
    mc_passes=10, mc_seed=1234,
    entropy_percentile=80.0, mel_recall_target=0.90,     # same threshold recipe as the training notebook
    bootstrap=1000,
    promote=dict(ham_macro_f1_drop=0.01, ham_mel_sens_drop=0.03, dark_false_alarm_rise=0.02, dark_hr_flag_drop=0.0),
    max_per_source=None, workers=16,
    num_workers=2 if os.path.exists("/kaggle/working") else 0,
)
if SMOKE:
    CFG.update(samples_per_epoch=16, batch_size=4, bootstrap=50, max_per_source=30, workers=8, mc_passes=3, dev_folds=2)
    CFG["base"]["epochs"] = 1; CFG["trials"] = [(n, {k: (1 if k == "epochs" else v) for k, v in o.items()}) for n, o in CFG["trials"][:3]]

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    (torch.ones(8, device=device) * 2).sum().item()
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU, training will be very slow")
CODES = CFG["class_codes"]; IDX = {c: i for i, c in enumerate(CODES)}
MEL = IDX["mel"]; HR_IDX = [IDX[c] for c in CFG["high_risk"]]

if IN_KAGGLE:
    OUT, WORK = "/kaggle/working/phase4b", "/tmp/work"
else:
    OUT = os.environ.get("PHASE4_OUT", "phase4b_out"); WORK = os.environ.get("PHASE4_WORK", "phase4_work")
os.makedirs(OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)

REPO = os.environ.get("PHASE4_REPO") or f"{WORK}/repo"
if not os.path.exists(f"{REPO}/backend/model/efficientnet_b3_best.pth"):
    subprocess.run(["git", "clone", "--depth", "1", CFG["repo_url"], REPO], check=True)
ORIG_CFG = json.load(open(f"{REPO}/backend/model/model_config.json"))
print("device:", device, "| smoke mode:", SMOKE, "| outputs:", OUT)

In [ ]:
# Small download helpers (retries, browser-like user agent, image check before saving)
UA = {"User-Agent": "Mozilla/5.0 (capstone research; non-commercial)"}

def http_json(url):
    for attempt in range(4):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r:
                return json.load(r)
        except Exception:
            if attempt == 3: raise
            time.sleep(2 * (attempt + 1))

def isic_search(params):
    """All results of an ISIC Archive image search (public API, no login)."""
    url = "https://api.isic-archive.com/api/v2/images/search/?" + urllib.parse.urlencode({**params, "limit": 100})
    out = []
    while url:
        page = http_json(url); out += page["results"]; url = page.get("next")
    return out

def download_one(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return True
    for attempt in range(3):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=30) as r:
                data = r.read()
            Image.open(io.BytesIO(data)).verify()              # reject HTML error pages / broken files
            with open(path, "wb") as f: f.write(data)
            return True
        except Exception:
            time.sleep(1 + attempt)
    return False

def download_many(df):
    if len(df) == 0:
        return df
    os.makedirs(os.path.dirname(df["path"].iloc[0]), exist_ok=True)
    with ThreadPoolExecutor(CFG["workers"]) as ex:
        ok = list(ex.map(lambda a: download_one(*a), zip(df["url"], df["path"])))
    print(f"   downloaded {sum(ok)} of {len(ok)}")
    return df[np.array(ok, dtype=bool)].reset_index(drop=True)

def limit(df):
    n = CFG["max_per_source"]
    return df if not n or len(df) <= n else df.sample(n, random_state=CFG["seed"]).reset_index(drop=True)

clin = lambda r, k: (r.get("metadata", {}).get("clinical") or {}).get(k)
ext_of = lambda url: os.path.splitext(urllib.parse.urlparse(url).path)[1] or ".jpg"

## 2. Data

| Source | Use | Licence |
|---|---|---|
| Fitzpatrick17k, Fitzpatrick IV-VI, labels mapped to our 7 classes | **train and test by 5-fold cross-validation** | CC BY-NC-SA 3.0 (Groh et al. 2021) |
| HAM10000 (Kaggle) | train (original frozen split), val for thresholds, test | CC BY-NC 4.0 |
| ISIC Archive, Fitzpatrick V-VI, Memorial Sloan Kettering, benign | **test only** (false alarms) | CC BY |

Fitzpatrick17k has no patient ids, so each image is its own group (duplicates removed by file hash). Images are never committed to the repo.

In [ ]:
# ---- Dark skin, labelled: Fitzpatrick17k IV-VI, labels that map to our classes (same mapping as Phase 3) ----
FITZ_MAP = {"melanoma": "mel", "superficial spreading melanoma ssm": "mel", "malignant melanoma": "mel",
            "lentigo maligna": "mel", "basal cell carcinoma": "bcc", "solid cystic basal cell carcinoma": "bcc",
            "basal cell carcinoma morpheiform": "bcc", "actinic keratosis": "akiec", "seborrheic keratosis": "bkl",
            "nevocytic nevus": "nv", "halo nevus": "nv", "congenital nevus": "nv", "dermatofibroma": "df",
            "pyogenic granuloma": "vasc", "granuloma pyogenic": "vasc"}
fz_all = pd.read_csv("https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv")
fz_all = fz_all[fz_all["label"].isin(FITZ_MAP)].drop_duplicates("md5hash")
def fitz_frame(d, name):
    return pd.DataFrame(dict(image_id=d["md5hash"], url=d["url"], path=[f"{WORK}/fitz/{h}.jpg" for h in d["md5hash"]],
                             dx=d["label"].map(FITZ_MAP), group=d["md5hash"], fitz=d["fitzpatrick_scale"], source=name))
dark = fitz_frame(fz_all[fz_all["fitzpatrick_scale"].isin(CFG["dark_fitz_types"])], "Fitzpatrick17k IV-VI")
print("Fitzpatrick17k IV-VI candidates:", len(dark), dark["dx"].value_counts().to_dict())
dark = download_many(limit(dark))
print("Fitzpatrick17k IV-VI usable:", len(dark), dark["dx"].value_counts().to_dict(), "| by type:", dark["fitz"].value_counts().to_dict())
fitz3 = download_many(limit(fitz_frame(fz_all[fz_all["fitzpatrick_scale"] == 3], "Fitzpatrick17k III (training only)")))
print("Fitzpatrick17k III usable (training pool only):", len(fitz3), fitz3["dx"].value_counts().to_dict())

# ---- Dark skin, benign (TEST ONLY): ISIC Archive images labelled Fitzpatrick V-VI (same set as Phase 3) ----
dark_recs = isic_search({"query": "fitzpatrick_skin_type:V OR fitzpatrick_skin_type:VI"})
rows = []
for r in sorted(dark_recs, key=lambda r: r["isic_id"]):
    if (r.get("attribution") or "").startswith("Memorial Sloan Kettering") and clin(r, "diagnosis_1") == "Benign":
        url = r["files"]["full"]["url"]
        rows.append(dict(image_id=r["isic_id"], url=url, path=f"{WORK}/dark_benign/{r['isic_id']}{ext_of(url)}",
                         dx=None, lesion=clin(r, "lesion_id"), group=clin(r, "patient_id"),
                         fitz=clin(r, "fitzpatrick_skin_type"), source="ISIC Fitzpatrick V-VI benign (MSK)"))
dark_benign = download_many(limit(pd.DataFrame(rows).drop_duplicates("lesion")))
print("dark-skin benign test set:", len(dark_benign), "lesions |", dark_benign["group"].nunique(), "patients")

In [ ]:
# ---- HAM10000 with the ORIGINAL frozen split (from the repo) ----
splits = {n: pd.read_csv(f"{REPO}/docs/results/split_{n}.csv") for n in ["train", "val", "test"]}
if IN_KAGGLE:
    import glob
    index = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob("/kaggle/input/**/*.jpg", recursive=True)}
    print("HAM10000 images found:", len(index))
    def ham(name):
        d = splits[name].copy(); d["path"] = d["image_id"].map(index)
        assert d["path"].notna().all(), "some HAM10000 images are missing"
        d["group"] = d["lesion_id"]; d["source"] = "HAM10000"; d["fitz"] = None
        return limit(d)
    ham_train, ham_val, ham_test = ham("train"), ham("val"), ham("test")
    orig_counts = pd.read_csv(f"{REPO}/docs/results/split_class_counts.csv").set_index("dx")
    same = all(int(splits[n]["dx"].value_counts().get(c, 0)) == int(orig_counts.loc[c, n]) for n in splits for c in orig_counts.index)
    assert same, "HAM10000 split does not match the original run: stop and check docs/results/split_*.csv"
else:
    # Local smoke test only: HAM10000 is not on this machine, so Fitzpatrick17k I-III images stand in for it.
    print("LOCAL SMOKE MODE: Fitzpatrick17k I-III images stand in for HAM10000 (results meaningless).")
    light = download_many(limit(fitz_frame(fz_all[fz_all["fitzpatrick_scale"].isin([1, 2, 3])], "HAM10000 (smoke stand-in)")))
    third = len(light) // 3
    ham_train, ham_val, ham_test = light.iloc[:third], light.iloc[third:2 * third], light.iloc[2 * third:]
for d in (ham_train, ham_val, ham_test, dark, fitz3):
    d["label"] = d["dx"].map(IDX).astype(int)
dark_benign["label"] = -1
print("HAM10000:", {k: len(v) for k, v in [("train", ham_train), ("val", ham_val), ("test", ham_test)]})

# ---- LOCKED test set: 20% of dark-skin images (stratified), half of the ISIC benign patients ----
n_lock = int(round(1 / CFG["lock_share"]))
sgkf = StratifiedGroupKFold(n_splits=n_lock, shuffle=True, random_state=CFG["seed"])
_, lock_idx = next(sgkf.split(dark, dark["dx"], groups=dark["group"]))
dark["split"] = "dev"; dark.loc[dark.index[lock_idx], "split"] = "locked_test"
dark_dev = dark[dark["split"] == "dev"].reset_index(drop=True)
dark_lock = dark[dark["split"] == "locked_test"].reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=CFG["dev_folds"], shuffle=True, random_state=CFG["seed"])
dark_dev["fold"] = -1
for k, (_, te) in enumerate(sgkf.split(dark_dev, dark_dev["dx"], groups=dark_dev["group"])):
    dark_dev.loc[te, "fold"] = k
assert (dark_dev["fold"] >= 0).all() and not set(dark_dev["image_id"]) & set(dark_lock["image_id"])
pats = sorted(dark_benign["group"].dropna().unique()); rng = np.random.default_rng(CFG["seed"]); rng.shuffle(pats)
lock_pats = set(pats[: len(pats) // 2])
benign_dev = dark_benign[~dark_benign["group"].isin(lock_pats)].reset_index(drop=True)
benign_lock = dark_benign[dark_benign["group"].isin(lock_pats)].reset_index(drop=True)
assert not set(benign_dev["group"]) & set(benign_lock["group"])
dev_pats = sorted(benign_dev["group"].unique()); fold_of = {p_: i % CFG["dev_folds"] for i, p_ in enumerate(dev_pats)}
benign_dev["fold"] = benign_dev["group"].map(fold_of).astype(int)          # benign CV folds by patient
# Fitzpatrick17k has duplicate photos under different hashes only rarely; drop any III image whose url is in the dark set
fitz3 = fitz3[~fitz3["url"].isin(dark["url"])].reset_index(drop=True)
manifest = dict(dark_dev=dict(n=len(dark_dev), classes=dark_dev["dx"].value_counts().to_dict(), folds=dark_dev["fold"].value_counts().to_dict()),
                dark_locked_test=dict(n=len(dark_lock), classes=dark_lock["dx"].value_counts().to_dict(),
                                      fitzpatrick=dark_lock["fitz"].value_counts().to_dict()),
                isic_benign_dev=dict(n=len(benign_dev), patients=int(benign_dev["group"].nunique())),
                isic_benign_locked_test=dict(n=len(benign_lock), patients=int(benign_lock["group"].nunique())),
                fitz3_training_pool=len(fitz3), ham=dict(train=len(ham_train), val=len(ham_val), test=len(ham_test)),
                licences={"HAM10000": "CC BY-NC 4.0", "Fitzpatrick17k": "CC BY-NC-SA 3.0 (Groh et al. 2021)",
                          "ISIC Fitzpatrick V-VI (MSK)": "CC BY (Memorial Sloan Kettering Cancer Center)"})
json.dump(manifest, open(f"{OUT}/phase4b_data_manifest.json", "w"), indent=2, default=str)
pd.concat([dark_dev.assign(split="dev"), dark_lock.assign(split="locked_test", fold=-1)])[
    ["image_id", "dx", "fitz", "split", "fold"]].to_csv(f"{OUT}/phase4b_dark_split.csv", index=False)
print(json.dumps(manifest, indent=1, default=str))

## 3. Model, inference and metrics
Same inference contract as the app: resize straight to 300x300, ImageNet normalisation, backbone once, classifier run 10 times with dropout on (MC Dropout), average the softmax.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((CFG["input_size"], CFG["input_size"])),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(30),
    transforms.RandomResizedCrop(CFG["input_size"], scale=(0.7, 1.0), ratio=(0.9, 1.1)),    # clinical photos are framed loosely
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.2),
    transforms.ToTensor(), transforms.Normalize(CFG["mean"], CFG["std"])])
eval_tf = transforms.Compose([transforms.Resize((CFG["input_size"], CFG["input_size"])),
                              transforms.ToTensor(), transforms.Normalize(CFG["mean"], CFG["std"])])

class ImgSet(Dataset):
    def __init__(self, df, tf): self.paths, self.labels, self.tf = df["path"].values, df["label"].values, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB")), int(self.labels[i])

def build_model():
    m = models.efficientnet_b3(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CODES))
    return m

def load_original():
    m = build_model()
    m.load_state_dict(torch.load(f"{REPO}/backend/model/efficientnet_b3_best.pth", map_location="cpu"))
    return m.to(device).eval()

@torch.no_grad()
def predict(model, df):
    """Returns (MC Dropout averaged probabilities, plain eval-mode probabilities)."""
    torch.manual_seed(CFG["mc_seed"])
    P_mc, P_plain = [], []
    model.eval()
    for x, _ in DataLoader(ImgSet(df, eval_tf), batch_size=32, shuffle=False, num_workers=CFG["num_workers"]):
        feats = F.adaptive_avg_pool2d(model.features(x.to(device)), 1).flatten(1)
        P_plain.append(F.softmax(model.classifier(feats), 1).cpu())
        model.classifier.train()                                   # dropout ON (classifier only)
        P_mc.append(torch.stack([F.softmax(model.classifier(feats), 1) for _ in range(CFG["mc_passes"])]).mean(0).cpu())
        model.classifier.eval()
    return torch.cat(P_mc).numpy(), torch.cat(P_plain).numpy()

entropy = lambda P: -(P * np.log(P + 1e-12)).sum(1)

def thresholds_from_val(P_val, y_val):
    """Same recipe as the training notebook, chosen on HAM10000 VALIDATION only."""
    ent = float(np.percentile(entropy(P_val), CFG["entropy_percentile"]))
    is_mel = y_val == MEL
    ok = [t for t in np.linspace(0.05, 0.95, 91) if is_mel.any() and (P_val[is_mel, MEL] >= t).mean() >= CFG["mel_recall_target"]]
    return ent, float(max(ok)) if ok else 0.05

def cluster_ci(fn, groups, n_boot):
    """95% CI by resampling whole groups (patients, lesions or images)."""
    groups = np.asarray(groups); uniq = np.unique(groups); pos = {g: np.where(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(CFG["seed"]); vals = []
    for _ in range(n_boot):
        idx = np.concatenate([pos[g] for g in rng.choice(uniq, len(uniq), replace=True)])
        v = fn(idx)
        if v is not None and np.isfinite(v): vals.append(v)
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def evaluate(P_mc, P_plain, df, ent_thr, mel_thr, flag=None):
    """Labelled sets: accuracy/F1/sensitivities. Benign-only set: false-alarm rates. All with 95% CIs.
    `flag` can be passed in when each image has its own thresholds (cross-validation)."""
    pred = P_plain.argmax(1)
    if flag is None: flag = (entropy(P_mc) >= ent_thr) | (P_mc[:, MEL] >= mel_thr)
    y = df["label"].values; g = df["group"].values; out = {"n": int(len(df)), "patients_or_groups": int(len(np.unique(g)))}
    def add(name, fn):
        v = fn(np.arange(len(y))); lo, hi = cluster_ci(fn, g, CFG["bootstrap"])
        out[name] = None if v is None else round(float(v), 4); out[name + "_ci95"] = [round(lo, 4), round(hi, 4)]
    if (y >= 0).all():
        labels = np.unique(y); hr_true = np.isin(y, HR_IDX)
        add("accuracy", lambda i: float((pred[i] == y[i]).mean()))
        add("macro_f1", lambda i: float(f1_score(y[i], pred[i], labels=labels, average="macro", zero_division=0)))
        add("melanoma_sensitivity", lambda i: float((pred[i][y[i] == MEL] == MEL).mean()) if (y[i] == MEL).any() else None)
        add("high_risk_sensitivity", lambda i: float(np.isin(pred[i][hr_true[i]], HR_IDX).mean()) if hr_true[i].any() else None)
        add("high_risk_sensitivity_with_flag",
            lambda i: float((np.isin(pred[i], HR_IDX) | flag[i])[hr_true[i]].mean()) if hr_true[i].any() else None)
        add("benign_false_alarm_rate",
            lambda i: float(np.isin(pred[i][~hr_true[i]], HR_IDX).mean()) if (~hr_true[i]).any() else None)
        add("review_flag_rate", lambda i: float(flag[i].mean()))
    else:
        add("false_alarm_high_risk_rate", lambda i: float(np.isin(pred[i], HR_IDX).mean()))
        add("false_alarm_melanoma_rate", lambda i: float((pred[i] == MEL).mean()))
        add("review_flag_rate", lambda i: float(flag[i].mean()))
    return out

## 4. Fine-tuning with a configurable recipe
Every recipe starts from the original weights. Each epoch draws its samples from three pools: HAM10000 train, the dark-skin training images (repeated with augmentation, since there are few) and, if the recipe asks for it, Fitzpatrick III photos. The number of epochs is fixed by the recipe, never chosen on held-out data.

In [ ]:
ham_counts = np.bincount(ham_train["label"].values, minlength=len(CODES)).astype(float); ham_counts[ham_counts == 0] = 1
class_w = torch.tensor(ham_counts.sum() / (len(CODES) * ham_counts), dtype=torch.float32)

BENIGN_IDX = [i for i in range(len(CODES)) if i not in HR_IDX]      # bkl, df, nv, vasc

def lesion_loss(logits, y, rc):
    """Cross-entropy for labelled images; for benign-only images (label -1): -log P(any benign class)."""
    lab = y >= 0; parts = []
    if lab.any():
        parts.append(F.cross_entropy(logits[lab], y[lab], weight=class_w.to(logits.device), label_smoothing=rc["label_smoothing"], reduction="none"))
    if (~lab).any():
        p_benign = F.softmax(logits[~lab].float(), 1)[:, BENIGN_IDX].sum(1)
        parts.append(-torch.log(p_benign.clamp_min(1e-6)))
    return torch.cat(parts).mean()

def finetune(dark_train, rc, tag, benign_train=None):
    ham_share = 1 - rc["dark_share"] - rc["fitz3_share"] - rc["benign_share"]
    assert ham_share > 0.05, "recipe leaves almost no HAM10000 in each epoch"
    pools = [(ham_train, ham_share, False), (dark_train, rc["dark_share"], rc["dark_class_balance"])]
    if rc["fitz3_share"] > 0 and len(fitz3): pools.append((fitz3, rc["fitz3_share"], False))
    if rc["benign_share"] > 0 and benign_train is not None and len(benign_train): pools.append((benign_train, rc["benign_share"], False))
    train_df = pd.concat([p[0] for p in pools], ignore_index=True); w = []
    for d, share, balance in pools:
        if balance:
            cnt = d["label"].map(d["label"].value_counts()).values.astype(float)
            ww = (1.0 / cnt); ww = ww / ww.sum() * share
        else:
            ww = np.full(len(d), share / max(len(d), 1))
        w.append(ww)
    w = np.concatenate(w)
    model = load_original().train()
    for blk in model.features[: rc["freeze_blocks"]]:
        for p in blk.parameters(): p.requires_grad = False
    opt = torch.optim.AdamW([{"params": [p for p in model.features.parameters() if p.requires_grad], "lr": rc["lr_backbone"]},
                             {"params": model.classifier.parameters(), "lr": rc["lr_head"]}], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=rc["epochs"])
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")
    loader = DataLoader(ImgSet(train_df, train_tf), batch_size=CFG["batch_size"], num_workers=CFG["num_workers"], drop_last=True,
                        sampler=WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=CFG["samples_per_epoch"], replacement=True))
    t0 = time.time()
    for ep in range(rc["epochs"]):
        model.train(); tot, n = 0.0, 0
        for blk in model.features[: rc["freeze_blocks"]]: blk.eval()      # frozen blocks keep their batch-norm statistics
        for x, y in loader:
            x, y = x.to(device), y.to(device); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                loss = lesion_loss(model(x), y, rc)
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); tot += float(loss) * len(y); n += len(y)
        sched.step()
    print(f"   [{tag}] {rc['epochs']} epochs, last loss {tot/max(n,1):.4f}, {time.time()-t0:.0f}s")
    model.eval()
    P_val = predict(model, ham_val)
    return model, thresholds_from_val(P_val[0], ham_val["label"].values), P_val

## 5. Original model: reference numbers on the development sets

In [ ]:
orig = load_original()
ORIG_THR = (ORIG_CFG["entropy_threshold"], ORIG_CFG["melanoma_flag_threshold"])
def quick(P, d, thr):
    """Fast scores for tuning (no bootstrap)."""
    pred = P[1].argmax(1); y = d["label"].values; flag = (entropy(P[0]) >= thr[0]) | (P[0][:, MEL] >= thr[1])
    hr = np.isin(y, HR_IDX) if (y >= 0).all() else None
    out = dict(accuracy=float((pred == y).mean()) if hr is not None else None,
               macro_f1=float(f1_score(y, pred, labels=np.unique(y), average="macro", zero_division=0)) if hr is not None else None,
               hr_sens_flag=float((np.isin(pred, HR_IDX) | flag)[hr].mean()) if hr is not None and hr.any() else None,
               mel_sens=float((pred[y == MEL] == MEL).mean()) if hr is not None and (y == MEL).any() else None,
               false_alarm=float(np.isin(pred, HR_IDX).mean()) if hr is None else None)
    return out
REF = dict(dark_dev=quick(predict(orig, dark_dev), dark_dev, ORIG_THR), ham_val=quick(predict(orig, ham_val), ham_val, ORIG_THR),
           benign_dev=quick(predict(orig, benign_dev), benign_dev, ORIG_THR))
print(json.dumps(REF, indent=1))
del orig; torch.cuda.empty_cache()

## 6. Tuning loop: one change at a time, scored by cross-validation on the development set

In [ ]:
nanmean = lambda v: float(np.nanmean([np.nan if x is None else x for x in v]))

def run_recipe(rc, name):
    oof_mc = np.zeros((len(dark_dev), len(CODES))); oof_plain = np.zeros_like(oof_mc); thr_img = np.zeros((len(dark_dev), 2))
    ham_f1, ham_mel, fa = [], [], []
    for k in range(CFG["dev_folds"]):
        m, thr, P_val = finetune(dark_dev[dark_dev["fold"] != k], rc, f"{name} fold{k}", benign_dev[benign_dev["fold"] != k])
        pos = np.where(dark_dev["fold"].values == k)[0]
        P_mc, P_plain = predict(m, dark_dev.iloc[pos]); oof_mc[pos], oof_plain[pos], thr_img[pos] = P_mc, P_plain, thr
        q = quick(P_val, ham_val, thr); ham_f1.append(q["macro_f1"]); ham_mel.append(q["mel_sens"])
        held = benign_dev[benign_dev["fold"] == k]
        if len(held): fa.append((len(held), quick(predict(m, held), held, thr)["false_alarm"]))
        del m; torch.cuda.empty_cache()
    flag = (entropy(oof_mc) >= thr_img[:, 0]) | (oof_mc[:, MEL] >= thr_img[:, 1])
    y = dark_dev["label"].values; pred = oof_plain.argmax(1); hr = np.isin(y, HR_IDX)
    return dict(dark_dev_macro_f1=float(f1_score(y, pred, labels=np.unique(y), average="macro", zero_division=0)),
                dark_dev_accuracy=float((pred == y).mean()), dark_dev_hr_sens_flag=float((np.isin(pred, HR_IDX) | flag)[hr].mean()),
                ham_val_macro_f1=nanmean(ham_f1), ham_val_mel_sens=nanmean(ham_mel),
                benign_dev_false_alarm=float(sum(n_ * v for n_, v in fa) / max(sum(n_ for n_, _ in fa), 1)))

def passes_guards(s):
    return (s["ham_val_macro_f1"] >= REF["ham_val"]["macro_f1"] - CFG["ham_val_f1_drop"] and
            s["benign_dev_false_alarm"] <= REF["benign_dev"]["false_alarm"] + CFG["dev_false_alarm_rise"])

log, started = [], time.time()
def log_trial(name, change, rc, s, kept):
    log.append(dict(trial=name, change=json.dumps(change), recipe=json.dumps(rc), kept=kept, guards_ok=passes_guards(s), **s,
                    minutes_since_start=round((time.time() - started) / 60, 1)))
    pd.DataFrame(log).to_csv(f"{OUT}/phase4b_tuning_log.csv", index=False)          # saved after every trial
    print(f"{'KEEP' if kept else 'drop'} | {name} | " + ", ".join(f"{k} {v:.3f}" for k, v in s.items()))

best = dict(CFG["base"]); best_s = run_recipe(best, "run-1 recipe"); log_trial("run-1 recipe (start)", {}, best, best_s, True)
for name, change in CFG["trials"]:
    if (time.time() - started) / 3600 > CFG["time_budget_h"]:
        print("time budget reached, stopping the loop"); break
    rc = {**best, **change}; s = run_recipe(rc, name)
    g_new, g_best = passes_guards(s), passes_guards(best_s)
    kept = (g_new and not g_best) or (g_new == g_best and s["dark_dev_macro_f1"] >= best_s["dark_dev_macro_f1"] + CFG["min_gain"])
    log_trial(name, change, rc, s, kept)
    if kept: best, best_s = rc, s
print("\nBEST RECIPE:", best); print("development scores:", best_s)

## 7. Final model and the one-time test on the locked sets

In [ ]:
final, FT_THR, _ = finetune(dark_dev, best, "final", benign_dev)
print(f"final thresholds (HAM10000 validation): entropy {FT_THR[0]:.4f}, melanoma {FT_THR[1]:.2f}")
orig = load_original()
vi = dark_lock["fitz"].isin([5, 6]).values
results = []
def record(model_name, set_name, P, d, thr):
    r = evaluate(P[0], P[1], d, *thr); r.update(model=model_name, eval_set=set_name); results.append(r)
    print(model_name, "|", set_name, {k: v for k, v in r.items() if not k.endswith("_ci95")})
for mname, m, thr in [("original", orig, ORIG_THR), ("fine-tuned (final)", final, FT_THR)]:
    P = predict(m, dark_lock)
    record(mname, "Locked dark skin IV-VI (Fitzpatrick17k)", P, dark_lock, thr)
    if vi.sum() >= 5: record(mname, "Locked dark skin V-VI only (Fitzpatrick17k)", (P[0][vi], P[1][vi]), dark_lock[vi], thr)
    record(mname, "Locked ISIC benign dark skin (MSK)", predict(m, benign_lock), benign_lock, thr)
    record(mname, "HAM10000 test", predict(m, ham_test), ham_test, thr)
res = pd.DataFrame(results); res.to_csv(f"{OUT}/phase4b_locked_test.csv", index=False)
show = [c for c in ["accuracy", "macro_f1", "melanoma_sensitivity", "high_risk_sensitivity_with_flag", "benign_false_alarm_rate",
                    "false_alarm_high_risk_rate", "review_flag_rate"] if c in res]
pd.set_option("display.width", 220); print(res.pivot_table(index="eval_set", columns="model", values=show, aggfunc="first").round(3))

## 8. Promotion rule (written before the run)
The final model replaces the app's model ONLY if all of these hold on the locked and original test sets:
1. HAM10000 test macro-F1 drops by no more than 0.01.
2. HAM10000 test melanoma sensitivity drops by no more than 0.03.
3. Locked dark-skin IV-VI accuracy is higher than the original model's.
4. Locked dark-skin IV-VI high-risk sensitivity counting the review flag does not drop.
5. False alarms on the locked ISIC benign dark-skin lesions rise by no more than 0.02.
Even if it passes, the author decides whether to deploy it.

In [ ]:
def get(model, s, m):
    row = res[(res.model == model) & (res.eval_set == s)]
    return float(row[m].iloc[0]) if len(row) and m in row and pd.notna(row[m].iloc[0]) else np.nan
P = CFG["promote"]; H, D, B, FT = "HAM10000 test", "Locked dark skin IV-VI (Fitzpatrick17k)", "Locked ISIC benign dark skin (MSK)", "fine-tuned (final)"
checks = {
    "HAM macro-F1 not worse":             get(FT, H, "macro_f1") >= get("original", H, "macro_f1") - P["ham_macro_f1_drop"],
    "HAM melanoma sens. not worse":       get(FT, H, "melanoma_sensitivity") >= get("original", H, "melanoma_sensitivity") - P["ham_mel_sens_drop"],
    "locked dark-skin accuracy improved": get(FT, D, "accuracy") > get("original", D, "accuracy"),
    "locked dark-skin high-risk catch not down": get(FT, D, "high_risk_sensitivity_with_flag") >= get("original", D, "high_risk_sensitivity_with_flag") - P["dark_hr_flag_drop"],
    "locked ISIC false alarms not up":    get(FT, B, "false_alarm_high_risk_rate") <= get("original", B, "false_alarm_high_risk_rate") + P["dark_false_alarm_rise"],
}
PROMOTE = all(bool(v) for v in checks.values()) and not SMOKE
for k, v in checks.items(): print(("PASS " if v else "FAIL ") + k)
print("\nPASSES PROMOTION RULE:", PROMOTE, "(smoke mode never promotes)" if SMOKE else "")

In [ ]:
def contract_metrics(model, d, thr):
    P_mc, P = predict(model, d); y = d["label"].values; pred = P.argmax(1)
    pr, rc_, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    try: auc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(len(CODES)))))
    except ValueError: auc = float("nan")
    flag = (entropy(P_mc) >= thr[0]) | (P_mc[:, MEL] >= thr[1]); ok = P_mc.argmax(1) == y; mel = y == MEL
    test = dict(accuracy=float((pred == y).mean()), macro_precision=float(pr), macro_recall=float(rc_), macro_f1=float(f1),
                macro_auc_ovr=auc, melanoma_sensitivity=float((pred[mel] == MEL).mean()) if mel.any() else None)
    mission = dict(mc_accuracy=float(ok.mean()), review_rate=float(flag.mean()),
                   accuracy_unflagged=float(ok[~flag].mean()) if (~flag).any() else None,
                   accuracy_flagged=float(ok[flag].mean()) if flag.any() else None,
                   errors_caught_by_flag=float(flag[~ok].mean()) if (~ok).any() else None,
                   melanoma_sensitivity_argmax=float(ok[mel].mean()) if mel.any() else None,
                   melanoma_sensitivity_with_flag=float((ok | flag)[mel].mean()) if mel.any() else None)
    return test, mission
test, mission = contract_metrics(final, ham_test, FT_THR)
summary = dict(passes_promotion_rule=PROMOTE, checks={k: bool(v) for k, v in checks.items()}, best_recipe=best, best_dev_scores=best_s,
               reference_dev_scores=REF, finetuned_thresholds=FT_THR, original_thresholds=ORIG_THR,
               config={k: v for k, v in CFG.items() if k != "repo_url"}, finetuned_ham_test_metrics=test, finetuned_ham_mission_metrics=mission,
               honest_scope="Dark-skin data is a small web-sourced Fitzpatrick17k sample (no patient ids, some label noise). The locked "
                            "test set is small; results are indicative only. Not a validation for clinical use on dark skin.")
json.dump(summary, open(f"{OUT}/phase4b_summary.json", "w"), indent=2, default=str)
# The candidate is always saved (so the author can decide), clearly labelled with whether it passed the rule
os.makedirs(f"{OUT}/model_candidate", exist_ok=True)
cand = {k: v for k, v in ORIG_CFG.items() if k not in ("temperature", "calibrated_entropy_threshold", "calibrated_melanoma_threshold", "calibration")}
cand.update(model_name="efficientnet_b3_finetuned_dark_skin_v4b", entropy_threshold=FT_THR[0], melanoma_flag_threshold=FT_THR[1],
            test_metrics=test, mission_metrics=mission, passes_promotion_rule=PROMOTE, recipe=best,
            training_data="HAM10000 + Fitzpatrick17k Fitzpatrick IV-VI development set" + (" + Fitzpatrick III" if best["fitz3_share"] > 0 else "")
                          + (" + ISIC benign dark-skin lesions (development patients)" if best["benign_share"] > 0 else ""),
            limitations="Fine-tuned with a small web-sourced dark-skin sample (Fitzpatrick17k). Dark-skin results come from a small "
                        "locked test set. Not validated for clinical use on dark skin.")
torch.save(final.state_dict(), f"{OUT}/model_candidate/efficientnet_b3_best.pth")
json.dump(cand, open(f"{OUT}/model_candidate/model_config.json", "w"), indent=2)
print("candidate saved to model_candidate/ (passes rule:", PROMOTE, ")")

In [ ]:
import matplotlib.pyplot as plt
lg = pd.DataFrame(log)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(len(lg)), lg["dark_dev_macro_f1"], "o-", label="dark-skin dev macro-F1 (CV)")
axes[0].plot(range(len(lg)), lg["ham_val_macro_f1"], "s--", label="HAM10000 val macro-F1")
axes[0].axhline(REF["dark_dev"]["macro_f1"], color="grey", ls=":", label="original model, dark-skin dev")
for i, k in enumerate(lg["kept"]): axes[0].annotate("kept" if k else "", (i, lg["dark_dev_macro_f1"].iloc[i]), fontsize=7)
axes[0].set_xticks(range(len(lg))); axes[0].set_xticklabels(lg["trial"], rotation=40, ha="right", fontsize=7)
axes[0].set_ylim(0, 1); axes[0].legend(fontsize=7); axes[0].set_title("Tuning steps (development data only)")
rows = [("Locked dark skin IV-VI (Fitzpatrick17k)", "accuracy"), ("Locked dark skin IV-VI (Fitzpatrick17k)", "macro_f1"),
        ("Locked dark skin IV-VI (Fitzpatrick17k)", "high_risk_sensitivity_with_flag"), ("HAM10000 test", "macro_f1"),
        ("Locked ISIC benign dark skin (MSK)", "false_alarm_high_risk_rate")]
xs = np.arange(len(rows))
for j, (mname, off) in enumerate([("original", -0.18), ("fine-tuned (final)", 0.18)]):
    v = [get(mname, s, m) for s, m in rows]
    ci = [res[(res.model == mname) & (res.eval_set == s)][m + "_ci95"].iloc[0] for s, m in rows]
    axes[1].bar(xs + off, v, 0.36, yerr=[[max(a - c[0], 0) for a, c in zip(v, ci)], [max(c[1] - a, 0) for a, c in zip(v, ci)]],
                capsize=3, label=mname)
NL = chr(10)
axes[1].set_xticks(xs); axes[1].set_xticklabels([s.replace("Locked ", "Locked" + NL) + NL + m for s, m in rows], fontsize=6)
axes[1].set_ylim(0, 1); axes[1].legend(fontsize=7); axes[1].set_title("One-time test on locked sets (95% CI)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_phase4b_tuning.png", dpi=150)
print("outputs:", sorted(os.listdir(OUT)))